<a href="https://colab.research.google.com/github/khushi92171/Python_project/blob/main/public_library_circulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
import pandas as pd      # pandas = tables (like Excel inside Python)
import numpy as np       # numpy  = maths (mean, median, percentiles)

FILE_NAME   = "11_library_circulation.csv"
COL_ID      = "Loan_ID"
COL_TITLE   = "Book_Title"
COL_GENRE   = "Genre"
COL_MEMBER  = "Member_Category"
COL_AGE     = "Borrower_Age"
COL_CHECKOUT = "Checkout_Date"
COL_RETURN  = "Return_Date"

LOAN_PERIOD = 14      # assumption: a book should be returned within 14 days
FINE_PER_DAY = 0.25   # assumption: fine of 0.25 per day late



In [52]:
# STEP 1: Load the file and look at it
print("STEP 1: LOAD DATA")
df = pd.read_csv(FILE_NAME)

print("Rows and columns:", df.shape)
print("Column names:", list(df.columns))
print("Data types:")
print(df.dtypes)
print(df.head())          # first 5 rows



STEP 1: LOAD DATA
Rows and columns: (1512, 7)
Column names: ['Loan_ID', 'Book_Title', 'Genre', 'Member_Category', 'Borrower_Age', 'Checkout_Date', 'Return_Date']
Data types:
Loan_ID             object
Book_Title          object
Genre               object
Member_Category     object
Borrower_Age       float64
Checkout_Date       object
Return_Date         object
dtype: object
   Loan_ID            Book_Title            Genre Member_Category  \
0  L100000   Big Bear's Birthday         Children         Student   
1  L100001    Signal from Kepler  Science Fiction           Adult   
2  L100002       Beyond the Wall      Young Adult         Student   
3  L100003  Rainbow Fish Friends         Children         Student   
4  L100004    Midnight in Mumbai          Fiction           Adult   

   Borrower_Age Checkout_Date Return_Date  
0          24.0    2025-06-27  2025-07-09  
1          34.0    2025-02-22  2025-03-04  
2           9.0    2025-07-28  2025-08-08  
3          21.0    2025-03-11  2

In [53]:
# STEP 2: Missing values and duplicates
print("\nSTEP 2: MISSING VALUES AND DUPLICATES")
print("Missing values in each column:")
print(df.isnull().sum())

duplicate_count = df[COL_ID].duplicated().sum()
print("Duplicate Loan_IDs:", duplicate_count)

# Keep only the first row for each Loan_ID
df = df.drop_duplicates(subset=COL_ID, keep="first")

# Clean text columns: remove extra spaces and make capitalisation the same
# (so "fiction", " Fiction" and "FICTION" count as ONE genre)
for col in [COL_TITLE, COL_GENRE, COL_MEMBER]:
    df[col] = df[col].astype(str).str.strip().str.title()




STEP 2: MISSING VALUES AND DUPLICATES
Missing values in each column:
Loan_ID              0
Book_Title           0
Genre                0
Member_Category      0
Borrower_Age        45
Checkout_Date        0
Return_Date        125
dtype: int64
Duplicate Loan_IDs: 12


In [54]:
# STEP 4 (done early because Step 3 needs dates): convert to dates
# errors="coerce" means: if a date is invalid, make it blank (NaT)
df[COL_CHECKOUT] = pd.to_datetime(df[COL_CHECKOUT], errors="coerce")
df[COL_RETURN]   = pd.to_datetime(df[COL_RETURN], errors="coerce")



In [55]:
# STEP 3: Handle missing values
print("\nSTEP 3: HANDLE MISSING VALUES")

# --- Return_Date ---
still_out = df[COL_RETURN].isnull().sum()
print("Books with no return date (still on loan):", still_out)

# --- Borrower_Age ---
# Ages below 0 or above 110 are impossible, so treat them as missing too.
df.loc[(df[COL_AGE] < 0) | (df[COL_AGE] > 110), COL_AGE] = np.nan
missing_ages = df[COL_AGE].isnull().sum()

# Fill missing ages with the MEDIAN age of the same member category.
# (Median is better than mean because a few extreme ages won't pull it.)
df[COL_AGE] = df[COL_AGE].fillna(df.groupby(COL_MEMBER)[COL_AGE].transform("median"))
# If a whole category had no ages, use the overall median instead
df[COL_AGE] = df[COL_AGE].fillna(df[COL_AGE].median())
print("Ages filled in:", missing_ages)




STEP 3: HANDLE MISSING VALUES
Books with no return date (still on loan): 124
Ages filled in: 45


In [56]:
# STEP 5: Loan duration in days
print("\nSTEP 5: LOAN DURATION")
df["Loan_Days"] = (df[COL_RETURN] - df[COL_CHECKOUT]).dt.days

# A return date BEFORE the checkout date is a data error -> make it blank
df.loc[df["Loan_Days"] < 0, "Loan_Days"] = np.nan

# Overdue = kept longer than the allowed loan period
df["Days_Late"] = df["Loan_Days"] - LOAN_PERIOD
df["Is_Overdue"] = df["Days_Late"] > 0       # True / False

# Make a smaller table with only books that were returned properly
returned = df[df["Loan_Days"].notnull()]
print("Returned loans with a valid duration:", len(returned))




STEP 5: LOAN DURATION
Returned loans with a valid duration: 1376


In [57]:
# STEP 6: Main numbers
print("\nSTEP 6: MAIN NUMBERS")
total_borrowed = len(df)
average_days = returned["Loan_Days"].mean()
overdue_percent = returned["Is_Overdue"].mean() * 100

print("Total books borrowed:", total_borrowed)
print("Average loan duration (days):", round(average_days, 2))
print("Overdue percentage:", round(overdue_percent, 2), "%")




STEP 6: MAIN NUMBERS
Total books borrowed: 1500
Average loan duration (days): 13.18
Overdue percentage: 26.45 %


In [58]:
# STEP 7: Top 10 most borrowed titles
print("\nSTEP 7: TOP 10 TITLES")
top_titles = df[COL_TITLE].value_counts().head(10)
print(top_titles)



STEP 7: TOP 10 TITLES
Book_Title
The Silent Garden              73
A River Called Home            68
Echoes Of Tomorrow             65
Paper Boats                    64
Pip The Little Fox             58
Cities And People              55
The Moon Who Wouldn'T Sleep    52
The Last Lighthouse            52
Midnight In Mumbai             51
The Habit Loop                 48
Name: count, dtype: int64


In [59]:
# STEP 8: Genres ranked by borrow count

print("\nSTEP 8: GENRE RANKING")
genre_counts = df[COL_GENRE].value_counts()
print(genre_counts)




STEP 8: GENRE RANKING
Genre
Fiction            373
Children           244
Non-Fiction        229
Mystery            173
Science Fiction    130
Romance            103
Young Adult        100
Self-Help           79
Biography           69
Name: count, dtype: int64


In [60]:
# STEP 9: Monthly borrowing and peak month
print("\nSTEP 9: MONTHLY BORROWING")
df["Month"] = df[COL_CHECKOUT].dt.to_period("M")     # e.g. 2025-03
monthly = df["Month"].value_counts().sort_index()    # sort by date
print(monthly)

print("Peak month:", monthly.idxmax(), "with", monthly.max(), "borrows")



STEP 9: MONTHLY BORROWING
Month
2025-01    117
2025-02     83
2025-03    117
2025-04    113
2025-05    110
2025-06    169
2025-07    180
2025-08    122
2025-09     84
2025-10    115
2025-11    100
2025-12    190
Freq: M, Name: count, dtype: int64
Peak month: 2025-12 with 190 borrows


In [61]:
# STEP 10: Borrowing by member category
print("\nSTEP 10: MEMBER CATEGORIES")
member_counts = df[COL_MEMBER].value_counts()
print(member_counts)



STEP 10: MEMBER CATEGORIES
Member_Category
Adult      638
Student    495
Senior     367
Name: count, dtype: int64


In [62]:
# STEP 11: Overdue rate by genre and by member category
# The mean of True/False gives the share that is True (e.g. 0.30 = 30%)
print("\nSTEP 11: OVERDUE RATES")
overdue_by_genre = returned.groupby(COL_GENRE)["Is_Overdue"].mean() * 100
print("Overdue % by genre:")
print(overdue_by_genre.sort_values(ascending=False).round(1))

overdue_by_member = returned.groupby(COL_MEMBER)["Is_Overdue"].mean() * 100
print("Overdue % by member category:")
print(overdue_by_member.sort_values(ascending=False).round(1))





STEP 11: OVERDUE RATES
Overdue % by genre:
Genre
Biography          38.1
Non-Fiction        35.3
Young Adult        31.9
Children           26.2
Romance            25.8
Fiction            23.7
Science Fiction    21.7
Mystery            21.3
Self-Help          16.7
Name: Is_Overdue, dtype: float64
Overdue % by member category:
Member_Category
Student    33.3
Adult      24.4
Senior     20.5
Name: Is_Overdue, dtype: float64


In [63]:
# STEP 12: Estimated late fines
print("\nSTEP 12: LATE FINES")
# Fine = days late x fine per day (only for overdue books)
df["Fine"] = 0.0
late_rows = df["Is_Overdue"] == True
df.loc[late_rows, "Fine"] = df.loc[late_rows, "Days_Late"] * FINE_PER_DAY

print("Total fines:", round(df["Fine"].sum(), 2))
print("Fines by member category:")
print(df.groupby(COL_MEMBER)["Fine"].sum().round(2))


STEP 12: LATE FINES
Total fines: 793.75
Fines by member category:
Member_Category
Adult      248.75
Senior     220.75
Student    324.25
Name: Fine, dtype: float64


In [64]:
# STEP 13: Unusually long loans (IQR method)
# Q1 = 25% mark, Q3 = 75% mark, IQR = Q3 - Q1
# Anything above Q3 + 1.5 x IQR is "unusually long"
print("\nSTEP 13: LONG LOANS (IQR METHOD)")
q1 = np.percentile(returned["Loan_Days"], 25)
q3 = np.percentile(returned["Loan_Days"], 75)
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr
print("Q1:", q1, " Q3:", q3, " IQR:", iqr)
print("Normal range:", lower_limit, "to", upper_limit)

outliers = returned[(returned["Loan_Days"] > upper_limit) |
                    (returned["Loan_Days"] < lower_limit)]
print("Number of unusual loans:", len(outliers))




STEP 13: LONG LOANS (IQR METHOD)
Q1: 8.0  Q3: 15.0  IQR: 7.0
Normal range: -2.5 to 25.5
Number of unusual loans: 33


In [65]:
# STEP 14: NumPy statistics
print("\nSTEP 14: NUMPY STATISTICS")
days = returned["Loan_Days"].to_numpy()
print("Mean:", round(np.mean(days), 2))
print("Median:", np.median(days))
print("Standard deviation:", round(np.std(days, ddof=1), 2))




STEP 14: NUMPY STATISTICS
Mean: 13.18
Median: 12.0
Standard deviation: 11.35


In [66]:
# STEP 15: Age groups
print("\nSTEP 15: AGE GROUPS")
age_limits = [0, 12, 17, 24, 39, 59, 110]
age_names = ["Child (0-12)", "Teen (13-17)", "Young adult (18-24)",
             "Adult (25-39)", "Middle age (40-59)", "Senior (60+)"]

# include_lowest=True makes sure age 0 is included
df["Age_Group"] = pd.cut(df[COL_AGE], bins=age_limits,
                         labels=age_names, include_lowest=True)
print(df["Age_Group"].value_counts().sort_index())



STEP 15: AGE GROUPS
Age_Group
Child (0-12)           154
Teen (13-17)           153
Young adult (18-24)    188
Adult (25-39)          270
Middle age (40-59)     368
Senior (60+)           367
Name: count, dtype: int64
